In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import difflib
import re

In [ ]:
runners = pd.read_excel("../../data/cleaned/results/runners.xlsx")

In [29]:
equipment = pd.read_excel("../../data/cleaned/acceptances/equipment.xlsx")

In [3]:
# 1. Get unique names and clean them to find potential duplicates
unique_names = equipment.horse_name.dropna().unique()

# 2. Look for close matches
similar_pairs = []
for i, name1 in enumerate(unique_names):
    # Get close matches from the rest of the list
    matches = difflib.get_close_matches(name1, unique_names[i+1:], n=3, cutoff=0.8)
    for match in matches:
        similar_pairs.append((name1, match))

# 3. View the results clearly
df_typos = pd.DataFrame(similar_pairs, columns=['Name A', 'Name B'])
df_typos

,Name A,Name B
0,POINT THE STAR,IN THE STARS
1,SPARKLING THEA,SPARKLING DEW
2,PRICELESSGIRL,PRICELESS GOLD
3,MASTER OF TRINI,MASTER OF UNIVE
4,MASTER OF TRINI,MASTER OF STUDI
...,...,...
230,STARLIGHTER,STARLIGHT DANCER
231,MAGNETAR,MAGENTA
232,QUEEN CARLA,QUEEN CAROLINE
233,GOLDEN HEART,GOLDEN HAMMER


In [5]:
unique_names = equipment.horse_name.dropna().unique()

# Create a DataFrame of unique names
df_names = pd.DataFrame({'original_name': unique_names})

# Create a 'clean' key: uppercase, no spaces, no punctuation
df_names['clean_key'] = df_names['original_name'].apply(
    lambda x: re.sub(r'[^A-Z0-9]', '', str(x).upper())
)

# Find clean keys that appear more than once (meaning they have variations)
duplicates = df_names[df_names.duplicated(subset=['clean_key'], keep=False)]

# Sort them so variations sit next to each other
duplicates.sort_values(by='clean_key')

,original_name,clean_key


In [33]:
# =========================
# 2. NORMALIZE
# =========================
def normalize(df):
    df = df.copy()
    df['meet_date'] = pd.to_datetime(df['meet_date'])
    
    def clean(x):
        x = str(x)
        x = x.replace('\xa0', ' ')          # kill non-breaking spaces
        x = x.strip().upper()
        x = re.sub(r'\s+', ' ', x)          # normalize spaces
        return x
    
    df['horse_name'] = df['horse_name'].apply(clean)
    return df

# =========================
# 3. BUILD RUNNER MAP
# =========================
runner_map = runners[['meet_date', 'horse_name', 'race_no']].drop_duplicates()

# group runners by date for faster lookup
runner_grouped = runner_map.groupby('meet_date')

# =========================
# 4. EXACT MATCH
# =========================
merged = equipment.merge(
    runner_map,
    on=['meet_date', 'horse_name'],
    how='left',
    indicator=True,
    suffixes=('_old', '_true')
)

exact_matched = merged[merged['_merge'] == 'both'].copy()
unmatched = merged[merged['_merge'] == 'left_only'].copy()

# =========================
# 5. PARTIAL MATCH (SAFE)
# =========================
resolved_rows = []
still_unmatched = []

for _, row in unmatched.iterrows():
    date = row['meet_date']
    name = row['horse_name']
    
    if date not in runner_grouped.groups:
        still_unmatched.append(row)
        continue
    
    candidates = runner_grouped.get_group(date)
    
    matches = candidates[
        candidates['horse_name'].apply(
            lambda x: x.startswith(name)
        )
    ]
    
    if len(matches) == 1:
        match = matches.iloc[0]
        row['race_no'] = match['race_no']
        row['horse_name'] = match['horse_name']
        resolved_rows.append(row)
    else:
        still_unmatched.append(row)

# convert lists to df
partial_matched = pd.DataFrame(resolved_rows)
edge_cases = pd.DataFrame(still_unmatched)

# =========================
# 6. CLEAN EXACT MATCH
# =========================
exact_matched['race_no'] = exact_matched['race_no_true']
exact_matched['horse_name'] = exact_matched['horse_name']  # safe since exact match

exact_matched = exact_matched.drop(
    columns=['_merge', 'race_no_old', 'race_no_true']
)

# =========================
# 7. FINAL COMBINE
# =========================
clean_equipment = pd.concat([exact_matched, partial_matched], ignore_index=True)

clean_equipment = clean_equipment.sort_values(
    by=['meet_date', 'race_no']
).reset_index(drop=True)

# =========================
# 8. EDGE CASES
# =========================
edge_case_list = edge_cases[['meet_date', 'horse_name']].drop_duplicates()

print("Recovered via partial match:", len(partial_matched))
print("Still unresolved:", len(edge_case_list))
display(edge_case_list.head())

# =========================
# 9. FORMAT + SAVE
# =========================
clean_equipment['meet_date'] = clean_equipment['meet_date'].dt.strftime('%Y-%m-%d')

clean_equipment.to_excel("../../data/cleaned/acceptances_cleaned/equipment.xlsx", index=False)

Recovered via partial match: 747
Still unresolved: 333


,meet_date,horse_name
24749,2026-03-15,THUNDERING PHOENIX
24750,2026-03-15,OPUS DEI
24751,2026-03-15,BREAK POINT
24752,2026-03-15,SON OF A GUN
24753,2026-03-15,NAMIRI


In [28]:
edge_case_list.to_csv("edge_case.csv")

In [21]:
for x in edge_case_list['horse_name'].head(10):
    print(repr(x))

'SUSSEX PRIDE'
'JAGER BOMB'
'GOLD BOND'
'AWESOME ONE'
'SASSY LASS'
'DOMINATION'
'TIMELESS DEEDS'
'AURODEN'
'MISS MONEYPENNY'
'TEXAS GOLD'


In [ ]:
# 1. Map out where the race number changes from the row above it
consecutive_groups = (equipment['race_no'] != equipment['race_no'].shift()).cumsum().values

# 2. Group by the array, the meet_date, and the race column
# This pulls meet_date into the resulting aggregation
race_counts = equipment.groupby([consecutive_groups, 'meet_date', 'race_no']).size().reset_index()

# 3. Rename columns cleanly to match the new structure
race_counts.columns = ['group_id', 'meet_date', 'race_no', 'count']

# 4. Filter for counts greater than 19 and display specific columns
filtered_counts = race_counts[race_counts['count'] > 19][['meet_date', 'race_no', 'count']]

with pd.option_context('display.max_rows', None):
    display(filtered_counts)

,meet_date,race_no,count
434,2011-02-06,184,22
437,2011-02-06,187,22
505,2011-03-06,255,22
668,2011-08-07,47,21
889,2011-11-27,34,20
1031,2012-02-04,176,22
1036,2012-02-04,181,20
1071,2012-02-19,216,20
1664,2013-02-02,171,20
1665,2013-02-02,172,20


In [ ]:
output_file = "../data/cleaned/acceptances_cleaned/acceptances.xlsx"

runners.to_excel(output_file, index=False)